# Taxonomy v0.12 against v0.11

**Goal:** compare v0.12 with v0.11 on the same 4 samples, with haiku and opus.

| | v0.11 | v0.12 |
|---|---|---|
| Recommendations | RECOMMENDATION with subkind `trade_idea` or `rating` | Two claim types: **TRADE** (executable; open/maintain/close/adjust) and **RATING** (scale; formal/allocation) |
| Argument field | `core_claim` | `conclusion` |
| Links between arguments | Implicit: a role reuses another argument's core claim | Explicit: a role references an argument ID (`A002`), allowed only if that argument has its own premises, assumptions or warrants |
| Roles | premises, conditions, qualifiers, concessions, risks, catalysts, implications, alternatives | Adds **assumptions** and **warrants**, both explicit only; conditions take SUPPOSITION only; risks, catalysts and concessions also take SUPPOSITION |
| Scenarios | `outcomes`; an empty scenario is a warning | `implications`; an empty scenario is an error; conditional clauses in claims raise a warning |
| Evidence checks | Exact substring only; every failure triggers a full regeneration | Deterministic repairs first: quote and dash folding, gaps of page furniture, fuzzy snapping, cue alignment, link canonicalisation |
| Coverage | Must-keep regex on exclusions only | Also warns when a must-keep sentence is neither cited nor excluded |

Pipeline (repo root): `.venv\Scripts\python.exe scripts\run_samples.py --limit 4`. It defaults to v0.12 and writes to `outputs/samples_v12`.

**Questions**
1. Validity, cost and repairs: do the deterministic repairs stop the evidence failures?
2. Trades and ratings: are stances and trades separated?
3. Hierarchy: are arguments referenced by ID, and only when they have reasoning?
4. Assumptions and warrants: are they rare and explicit?
5. Conditionals and scenarios: do consequents move into scenario implications?
6. Atomicity: fewer duplicates and less data restated in claims?
7. Coverage: are v0.11 claims kept, and are must-keep sentences accounted for?
8. Do haiku and opus agree better?

The probes in this notebook are heuristics.

In [ ]:
import json
import re
import sys
from collections import Counter
from itertools import combinations
from pathlib import Path

import pandas as pd
from IPython.display import display

from incolor.argumentation.evaluate import match_statements, span_overlap
from incolor.argumentation.grounding import coverage
from incolor.argumentation.taxonomy import PACKAGE_DIR, load_taxonomy
from incolor.argumentation.validate import ARGUMENT_ROLES, argument_edges, is_argument_ref, scenario_edges
from incolor.argumentation.writer import load_argument_map

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RUN_DIRS = {"v0.11": ROOT / "outputs" / "samples_v11", "v0.12": ROOT / "outputs" / "samples_v12"}
COMPARED = list(RUN_DIRS)
MODELS = ["haiku", "opus"]
REC_TYPES = {"RECOMMENDATION", "TRADE", "RATING"}
TYPES = ["RECOMMENDATION", "TRADE", "RATING", "PREDICTION", "CAUSAL", "VIEW", "SUPPOSITION"]
V12 = load_taxonomy(PACKAGE_DIR / "prompts" / "taxonomy_v12.yaml")
JSON_COLUMNS = ["evidence_ids", "evidence_texts", "roles", "argument_ids"]
sys.path.insert(0, str(ROOT / "scripts"))
import build_claims_csv  # noqa: E402


def load_claims(runs_dir):
    """Statement table for one batch, rebuilt when the set of valid maps has changed."""
    data_dir = runs_dir / "claims_analysis"
    path = data_dir / "claims_all.csv"
    valid = {p.relative_to(ROOT).as_posix() for p in runs_dir.glob("*/*/argument_map.yaml")}
    built = set(pd.read_csv(path, usecols=["map_path"], encoding="utf-8-sig")["map_path"]) if path.exists() else set()
    if built != valid:
        build_claims_csv.main([
            "--runs-root", str(runs_dir), "--texts-dir", str(runs_dir / "inputs"), "--no-gold",
            "--pivot-model", "opus", "--out", str(data_dir),
        ])
    df = pd.read_csv(path, encoding="utf-8-sig")
    for col in JSON_COLUMNS:
        df[col] = df[col].map(lambda v: json.loads(v) if isinstance(v, str) and v else [])
    df["sub"] = df["subkinds"].fillna("").str.replace("|", "+")
    return df


DOCS = sorted({p.parent.parent.name for p in RUN_DIRS["v0.12"].glob("*/*/summary.json")})
LABEL = {doc: doc.split("_-_")[0].replace("__", " ").replace("_", " ")[:24] for doc in DOCS}
claims = pd.concat({v: load_claims(RUN_DIRS[v]) for v in COMPARED}, names=["taxonomy"]).reset_index(level=0)
claims = claims[claims["doc"].isin(DOCS)].reset_index(drop=True)
claims["label"] = claims["doc"].map(LABEL)
maps = {
    v: {(p.parent.parent.name, p.parent.name): load_argument_map(p) for p in sorted(RUN_DIRS[v].glob("*/*/argument_map.yaml"))
        if p.parent.parent.name in DOCS}
    for v in COMPARED
}
SOURCES = {doc: (RUN_DIRS["v0.12"] / "inputs" / f"{doc}.txt").read_text(encoding="utf-8") for doc in DOCS}
SHOW = ["taxonomy", "label", "model", "statement_id", "claim_type", "sub", "modality", "statement"]
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)
pd.crosstab([claims["label"], claims["model"]], claims["taxonomy"]).reindex(columns=COMPARED, fill_value=0)

## 1. Runs: validity, cost and repairs

Issue codes are those of the final attempt. In v0.12 the repair codes (`EVIDENCE_SNAPPED`, `EXCLUSION_SNAPPED`, `CUE_SNAPPED`, `LINK_CANONICALISED`) are warnings that record a deterministic fix.

In [ ]:
def load_runs(runs_dir):
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(runs_dir.glob("*/*/summary.json"))]
    runs = pd.DataFrame(rows)
    runs = runs[runs["doc"].isin(DOCS)].copy()
    runs["label"] = runs["doc"].map(LABEL)
    return runs


runs = pd.concat({v: load_runs(d) for v, d in RUN_DIRS.items()}, names=["taxonomy"]).reset_index(level=0)
RUN_COLS = ["ok", "attempts", "cost_usd", "seconds", "arguments", "statements", "evidence", "edges", "isolated", "exclusions"]
display(
    runs.pivot_table(index=["label", "model"], columns="taxonomy", values=RUN_COLS, aggfunc="first")
    .reindex(columns=COMPARED, level=1)
)
totals = runs.groupby(["taxonomy", "model"]).agg(
    runs=("doc", "size"), valid=("ok", "sum"), cost_usd=("cost_usd", "sum"), mean_attempts=("attempts", "mean"),
    statements=("statements", "sum"), evidence=("evidence", "sum"), arguments=("arguments", "sum"),
    exclusions=("exclusions", "sum"), median_seconds=("seconds", "median"),
).round(2)
display(totals)
display(runs[~runs["ok"].fillna(False).astype(bool)][["taxonomy", "label", "model", "attempts", "errors", "issue_codes"]])
codes = runs.set_index(["taxonomy", "model"])["issue_codes"].dropna().apply(pd.Series)
codes.groupby(level=[0, 1]).sum().T.fillna(0).astype(int)

In [ ]:
from incolor.argumentation.pipeline import _extract_json
from incolor.argumentation.repair import repair_output
from incolor.argumentation.schema import ModelOutput
from incolor.argumentation.validate import validate

# Every v0.12 attempt, validated as the model wrote it and after the deterministic repairs.
rows = []
for raw_path in sorted(RUN_DIRS["v0.12"].glob("*/*/raw_attempt*.txt")):
    doc, model = raw_path.parent.parent.name, raw_path.parent.name
    try:
        output = ModelOutput.model_validate(json.loads(_extract_json(raw_path.read_text(encoding="utf-8"))))
    except ValueError:
        rows.append({"label": LABEL[doc], "model": model, "attempt": int(raw_path.stem[-1]), "parsed": False})
        continue
    raw_errors = Counter(i.code for i in validate(output, SOURCES[doc], V12) if i.severity == "error")
    repaired, notes = repair_output(output, SOURCES[doc], V12)
    errors = Counter(i.code for i in validate(repaired, SOURCES[doc], V12) if i.severity == "error")
    methods = Counter(n.message.split(" by ")[1].split()[0] for n in notes if n.code == "EVIDENCE_SNAPPED")
    rows.append({
        "label": LABEL[doc], "model": model, "attempt": int(raw_path.stem[-1]), "parsed": True,
        "verbatim_errors_raw": raw_errors["EVIDENCE_NOT_VERBATIM"], "errors_raw": sum(raw_errors.values()),
        "errors_after_repair": sum(errors.values()), "snapped": dict(methods),
        "cues_fixed": sum(n.code == "CUE_SNAPPED" for n in notes),
        "links_canonicalised": sum(n.code == "LINK_CANONICALISED" for n in notes),
        "remaining": dict(errors),
    })
attempts = pd.DataFrame(rows)
attempts

### Observations

- 

## 2. Claim types: trades and ratings

In [ ]:
display(
    pd.crosstab([claims["taxonomy"], claims["model"]], claims["claim_type"], margins=True)
    .reindex(columns=[*TYPES, "All"], fill_value=0)
)
recs = claims[claims["claim_type"].isin(REC_TYPES)]
display(pd.crosstab([recs["taxonomy"], recs["model"]], recs["claim_type"] + ":" + recs["sub"]))
recs[recs["taxonomy"] == "v0.12"].sort_values(["label", "model", "statement_id"])[SHOW + ["attribution", "role_set"]]

### Observations

- 

## 3. Hierarchy: argument references

`id_links` are links made by argument ID (v0.12 only); `arg_links` also counts a conclusion reused as a statement. A `top_level` argument is not used by any other argument. A trade or rating is `motivated` when its premises reference another argument, another argument's conclusion, or a RATING.

In [ ]:
REASONING = ("premises", "assumptions", "warrants", "supporting")


def depth(edges):
    graph = {}
    for src, _, tgt in edges:
        graph.setdefault(src, set()).add(tgt)

    def walk(node, seen):
        return 0 if node in seen else 1 + max((walk(n, seen | {node}) for n in graph.get(node, ())), default=0)

    return max((walk(n, frozenset()) for n in graph), default=0)


def hierarchy_stats(v):
    rows = []
    for (doc, model), m in maps[v].items():
        types = {s.id: s.claim_type for s in m.statements}
        ids = {a.id for a in m.arguments}
        conclusions = {a.conclusion for a in m.arguments}
        edges, sedges = argument_edges(m), scenario_edges(m)
        sources = {e[0] for e in edges}
        linked = sources | {e[2] for e in edges} | {e[2] for e in sedges}
        refs = [(a, role, r) for a in m.arguments for role in ARGUMENT_ROLES for r in getattr(a, role)]
        recs = [a for a in m.arguments if types[a.conclusion] in REC_TYPES]
        rows.append({
            "taxonomy": v, "label": LABEL[doc], "model": model, "arguments": len(m.arguments),
            "arg_links": len(edges), "id_links": sum(is_argument_ref(r) for _, _, r in refs),
            "scenario_links": len(sedges), "max_depth": depth(edges),
            "isolated": sum(a.id not in linked for a in m.arguments),
            "top_level": sum(a.id not in sources for a in m.arguments),
            "bare": sum(not any(getattr(a, role) for role in ARGUMENT_ROLES) for a in m.arguments),
            "reasoned": sum(any(getattr(a, role) for role in REASONING) for a in m.arguments),
            "recs": len(recs),
            "recs_with_premises": sum(bool(a.premises or a.supporting) for a in recs),
            "recs_motivated": sum(
                any(r in ids or r in conclusions or types.get(r) == "RATING" for r in a.premises + a.supporting)
                for a in recs
            ),
        })
    return pd.DataFrame(rows)


hierarchy = pd.concat([hierarchy_stats(v) for v in COMPARED], ignore_index=True)
display(hierarchy.groupby(["taxonomy", "model"]).sum(numeric_only=True))
display(hierarchy.pivot_table(index=["label", "model"], columns="taxonomy",
                              values=["arguments", "arg_links", "id_links", "isolated", "top_level", "max_depth"],
                              aggfunc="first").reindex(columns=COMPARED, level=1))
# Roles used per version: counts of role members.
role_use = Counter()
for v in COMPARED:
    for (doc, model), m in maps[v].items():
        for a in m.arguments:
            for role in ARGUMENT_ROLES:
                role_use[(v, model, role)] += len(getattr(a, role))
pd.Series(role_use).unstack(level=2).fillna(0).astype(int).loc[:, lambda d: d.sum() > 0]

In [ ]:
# Links made by argument ID in v0.12, with the conclusions they join.
rows = []
for (doc, model), m in maps["v0.12"].items():
    st = {s.id: s for s in m.statements}
    by_id = {a.id: a for a in m.arguments}
    for a in m.arguments:
        for role in ARGUMENT_ROLES:
            for r in getattr(a, role):
                if is_argument_ref(r):
                    src = st[by_id[r].conclusion]
                    rows.append({"label": LABEL[doc], "model": model, "from": r, "role": role, "to": a.id,
                                 "from_type": src.claim_type, "from_brief": src.brief,
                                 "to_type": st[a.conclusion].claim_type, "to_brief": st[a.conclusion].brief})
id_links = pd.DataFrame(rows)
display(id_links.groupby(["model", "role"]).size().unstack(fill_value=0) if len(id_links) else "no ID links")
id_links

### Observations

- 

## 4. Assumptions and warrants

Both should be rare and explicit. Every one is listed with its evidence, for review. `source_mentions` counts the sentences in the sources that use adoption wording (assume, base case, predicated on), as an upper bound for explicit assumptions.

In [ ]:
ADOPTION = re.compile(V12.roles["assumptions"]["evidence_cue"]["pattern"])
from incolor.argumentation.grounding import sentence_spans, squash  # noqa: E402

mentions = {}
for doc in DOCS:
    text = squash(SOURCES[doc])
    mentions[LABEL[doc]] = [text[a:b] for a, b in sentence_spans(text) if ADOPTION.search(text[a:b])]
print({label: len(found) for label, found in mentions.items()}, "sentences with adoption wording")

rows = []
for (doc, model), m in maps["v0.12"].items():
    st = {s.id: s for s in m.statements}
    ev = {e.id: e.text for e in m.evidence}
    for a in m.arguments:
        for role in ("assumptions", "warrants"):
            for sid in getattr(a, role):
                s = st[sid]
                rows.append({"label": LABEL[doc], "model": model, "role": role, "argument": a.id,
                             "conclusion": st[a.conclusion].brief, "type": s.claim_type, "statement": s.statement,
                             "evidence": " | ".join(ev[e] for e in s.evidence)})
explicit = pd.DataFrame(rows)
explicit

### Observations

- 

## 5. Conditionals and scenarios

Claims other than suppositions that still carry a conditional clause, using the v0.12 `conditional_pattern` on both versions; then the scenarios and how they are referenced.

In [ ]:
CONDITIONAL = re.compile(V12.conditional_pattern)
cond = claims[(claims["claim_type"] != "SUPPOSITION") & claims["statement"].map(lambda s: bool(CONDITIONAL.search(s)))]
display(pd.crosstab(cond["taxonomy"], cond["model"]).reindex(COMPARED, fill_value=0))
display(cond[cond["taxonomy"] == "v0.12"][SHOW])
sup = claims[claims["claim_type"] == "SUPPOSITION"]
display(pd.crosstab([sup["taxonomy"], sup["model"]], sup["role_set"].fillna("") + " " +
                    sup["scenario_slots"].fillna("").str.replace(r"SC\d+:", "", regex=True)))
rows = []
for v in COMPARED:
    for (doc, model), m in maps[v].items():
        refs = Counter(role for _, role, _ in scenario_edges(m))
        rows.append({"taxonomy": v, "label": LABEL[doc], "model": model, "scenarios": len(m.scenarios),
                     "implications": sum(len(s.implications) for s in m.scenarios),
                     "labels": "; ".join(s.label for s in m.scenarios), "referenced_from": dict(refs)})
pd.DataFrame(rows).query("scenarios > 0")

### Observations

- 

## 6. Atomicity and duplication

Near-duplicate pairs within a map: two claims (not suppositions) whose content words overlap with a Jaccard score of at least 0.4, per 10 claims. Statement shape: share of statements with more than one sentence, and share of VIEW or CAUSAL statements that restate a number (data that belongs in evidence). Then the BofA case: claims on "France risk is priced".

In [ ]:
STOP = set(
    "the a an and or of to in on for with by at from is are was were be been will would could may might should "
    "that this these those it its as than more less into over under after before also has have had not no".split()
)
MULTI_SENTENCE = re.compile(r"[.!?]\s+[A-Z]")


def content_words(text):
    return {w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP and len(w) > 2}


rows, pairs = [], []
for v in COMPARED:
    for (doc, model), m in maps[v].items():
        claims_ = [s for s in m.statements if s.claim_type != "SUPPOSITION"]
        words = {s.id: content_words(s.statement) for s in claims_}
        dup = 0
        for a, b in combinations(claims_, 2):
            wa, wb = words[a.id], words[b.id]
            score = len(wa & wb) / len(wa | wb) if wa | wb else 0
            if score >= 0.4:
                dup += 1
                pairs.append({"taxonomy": v, "label": LABEL[doc], "model": model, "jaccard": round(score, 2),
                              "a": a.statement, "b": b.statement})
        judged = [s for s in claims_ if s.claim_type in ("VIEW", "CAUSAL")]
        rows.append({
            "taxonomy": v, "label": LABEL[doc], "model": model, "claims": len(claims_), "near_dup_pairs": dup,
            "multi_sentence": sum(bool(MULTI_SENTENCE.search(s.statement)) for s in m.statements),
            "view_causal": len(judged), "view_causal_with_number": sum(bool(re.search(r"\d", s.statement)) for s in judged),
            "mean_chars": round(sum(len(s.statement) for s in m.statements) / len(m.statements)),
        })
shape = pd.DataFrame(rows)
pairs = pd.DataFrame(pairs)
summary = shape.groupby(["taxonomy", "model"]).sum(numeric_only=True)
summary["pairs_per_10"] = (10 * summary["near_dup_pairs"] / summary["claims"]).round(2)
summary["multi_sentence_share"] = (summary["multi_sentence"] / summary["claims"]).round(2)
summary["number_share"] = (summary["view_causal_with_number"] / summary["view_causal"]).round(2)
display(summary.drop(columns="mean_chars"))
display(shape.pivot_table(index=["label", "model"], columns="taxonomy", values=["claims", "near_dup_pairs", "mean_chars"],
                          aggfunc="first").reindex(columns=COMPARED, level=1))
pairs[pairs["taxonomy"] == "v0.12"].sort_values("jaccard", ascending=False).head(12)

In [ ]:
# The BofA case: statements about France risk being priced, or the forward-vol premium.
bofa = claims[claims["label"].str.startswith("BofA Liquid") & claims["statement"].str.contains(
    r"priced|pricing|premium|reflected", case=False, regex=True)]
bofa.sort_values(["taxonomy", "model", "statement_id"])[SHOW + ["n_evidence"]]

### Observations

- 

## 7. Coverage

First, every source sentence is classed as cited, excluded or silent, and must-keep sentences (trade, rating, prediction, valuation or positioning wording) are counted. Then each v0.11 map is aligned to the v0.12 map of the same sample and model. A v0.11 claim is `matched` one-to-one (shared evidence, or statement wording with token F1 of at least 0.6), or `covered` when its evidence is still cited by some v0.12 statement.

In [ ]:
rows = []
for v in COMPARED:
    for (doc, model), m in maps[v].items():
        c = coverage(m, SOURCES[doc], V12.must_keep_patterns)
        rows.append({"taxonomy": v, "label": LABEL[doc], "model": model, **c.as_dict(),
                     "silent_examples": [s[:100] for s in c.must_keep_silent[:3]]})
sentence_cov = pd.DataFrame(rows)
cov_totals = sentence_cov.groupby(["taxonomy", "model"])[["sentences", "cited", "excluded", "silent", "must_keep",
                                                          "must_keep_silent"]].sum()
cov_totals["silent_share"] = (cov_totals["silent"] / cov_totals["sentences"]).round(2)
display(cov_totals)
display(sentence_cov[sentence_cov["taxonomy"] == "v0.12"][["label", "model", "must_keep_silent", "silent_examples"]])

rows = []
for (doc, model), m12 in maps["v0.12"].items():
    m11 = maps["v0.11"].get((doc, model))
    if m11 is None:
        continue
    new = {s.id: s for s in m12.statements}
    mapping = match_statements(m11, m12, text_only_threshold=0.6)
    ev11 = {e.id: e.text for e in m11.evidence}
    cited12 = [e.text for e in m12.evidence if any(e.id in s.evidence for s in m12.statements)]
    for s in m11.statements:
        n = new.get(mapping.get(s.id))
        rows.append({
            "label": LABEL[doc], "model": model, "v11_type": s.claim_type, "v11_sub": "+".join(s.subkinds),
            "v11_statement": s.statement, "v12_type": n.claim_type if n else None,
            "v12_statement": n.statement if n else None, "matched": n is not None,
            "covered": all(any(span_overlap(ev11[eid], t) >= 0.8 for t in cited12) for eid in s.evidence),
        })
align = pd.DataFrame(rows)
align["must_keep"] = align["v11_type"].isin(["RECOMMENDATION", "PREDICTION"]) | align["v11_sub"].isin(
    ["valuation", "positioning"])
display(align.groupby(["label", "model"])[["matched", "covered"]].mean().round(2)
        .join(align.groupby(["label", "model"]).size().rename("v11_claims")))
display(align.groupby("v11_type")[["matched", "covered"]].mean().round(2).join(align.groupby("v11_type").size().rename("n")))
display(pd.crosstab(align[align["v11_type"] == "RECOMMENDATION"]["v11_sub"],
                    align[align["v11_type"] == "RECOMMENDATION"]["v12_type"].fillna("unmatched")))
print("v0.11 must-keep claims neither matched nor covered (candidate losses)")
align[align["must_keep"] & ~align["matched"] & ~align["covered"]][["label", "model", "v11_type", "v11_sub", "v11_statement"]]

### Observations

- 

## 8. Model invariance: haiku vs opus

Statements are matched between the haiku and opus maps of each sample (shared evidence, as in v0.11, so the versions are comparable). `type_family` treats RECOMMENDATION, TRADE and RATING as one type.

In [ ]:
def family(t):
    return "REC" if t in REC_TYPES else t


rows = []
for v in COMPARED:
    for doc in DOCS:
        if not all((doc, model) in maps[v] for model in MODELS):
            continue
        a, b = maps[v][(doc, "haiku")], maps[v][(doc, "opus")]
        mapping = match_statements(a, b)
        sa, sb = {s.id: s for s in a.statements}, {s.id: s for s in b.statements}
        pairs_ = [(sa[x], sb[y]) for x, y in mapping.items()]
        n = len(pairs_)
        share = lambda test, pairs_=pairs_, n=n: round(sum(test(x, y) for x, y in pairs_) / n, 2) if n else None
        rows.append({
            "taxonomy": v, "label": LABEL[doc], "haiku_claims": len(sa), "opus_claims": len(sb), "matched": n,
            "claim_overlap_f1": round(2 * n / (len(sa) + len(sb)), 2),
            "type": share(lambda x, y: x.claim_type == y.claim_type),
            "type_family": share(lambda x, y: family(x.claim_type) == family(y.claim_type)),
            "subkinds": share(lambda x, y: set(x.subkinds) == set(y.subkinds)),
            "modality": share(lambda x, y: x.modality == y.modality),
        })
agreement = pd.DataFrame(rows)
agreement.pivot_table(index="label", columns="taxonomy",
                      values=["haiku_claims", "opus_claims", "claim_overlap_f1", "type", "type_family"],
                      aggfunc="first").reindex(columns=COMPARED, level=1)

### Observations

- 

## 9. Summary

- 